# Credit Risk Feature Engineering

## Objective

This notebook transforms the raw Home Credit application data into a modeling-ready dataset.

The objectives are to:

1. Split the data before learning preprocessing parameters
2. Correct invalid and special values
3. Create interpretable credit-risk features
4. Define numerical and categorical feature groups
5. Prepare a reproducible preprocessing strategy
6. Avoid target leakage

This notebook focuses on feature engineering and preprocessing design. Model training and evaluation will be completed in the next notebook.

## Modeling Principles

The feature-engineering workflow follows several important principles:

- Split the data before fitting imputers, encoders, or scalers
- Use only information available at application time
- Preserve missingness when it may carry risk information
- Prefer interpretable financial ratios over arbitrary transformations
- Handle division-by-zero explicitly
- Keep raw variables when engineered features provide complementary information
- Apply identical transformations to training and test data

## 1. Setup and Data Loading

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 100)

print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: /Users/hxxy/Desktop/找工/credit-risk-decisioning/.venv/bin/python
Python version: 3.13.1 (main, Dec  3 2024, 17:59:52) [Clang 16.0.0 (clang-1600.0.26.4)]


In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "application_train.csv"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

### Basic Validation

Before feature engineering, verify the target, identifier, and dataset grain.

In [3]:
required_columns = ["SK_ID_CURR", "TARGET"]

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Missing required columns: {missing_required_columns}"
    )

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique applicants:", df["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", df["SK_ID_CURR"].duplicated().sum())
print("\nTarget distribution:")
print(df["TARGET"].value_counts(dropna=False))
print("\nTarget rate:")
print(df["TARGET"].mean())

Rows: 307511
Columns: 122
Unique applicants: 307511
Duplicate applicant IDs: 0

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Target rate:
0.08072881945686496


### Observation

- The application table contains one row per applicant.
- `SK_ID_CURR` is an identifier and should not be used as a model feature.
- `TARGET = 1` represents applicants with repayment difficulties.
- The target is imbalanced, so model evaluation should not rely on accuracy alone.

## 2. Train/Test Split

The dataset is split before fitting imputers, encoders, scalers, or other preprocessing objects.

A stratified split is used to preserve the target distribution in both datasets.

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype("int8")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target rate:")
print(y_train.mean())

print("\nTest target rate:")
print(y_test.mean())

X_train shape: (246008, 121)
X_test shape: (61503, 121)

Training target rate:
0.08072908198107379

Test target rate:
0.08072776937710356


In [5]:
assert len(X_train) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(set(X_test.index))
assert abs(y_train.mean() - y_test.mean()) < 0.001

print("Train/test split validation passed.")

Train/test split validation passed.


In [6]:
train_ids = X_train["SK_ID_CURR"].copy()
test_ids = X_test["SK_ID_CURR"].copy()

X_train = X_train.drop(columns=["SK_ID_CURR"])
X_test = X_test.drop(columns=["SK_ID_CURR"])

print("Modeling training shape:", X_train.shape)
print("Modeling test shape:", X_test.shape)

Modeling training shape: (246008, 120)
Modeling test shape: (61503, 120)


## 3. Initial Feature Scope

The first feature-engineering version focuses on variables that are:

- available at application time,
- interpretable to business stakeholders,
- supported by the EDA and data dictionary,
- suitable for an interpretable baseline model.

Additional variables can be added after the baseline workflow is validated.

In [7]:
initial_features = [
    # Application and demographics
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    
    # Income and credit
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    
    # Employment and applicant profile
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    
    # Time-based variables
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    
    # Geographic / application context
    "REGION_POPULATION_RELATIVE",
    "REGION_RATING_CLIENT",
    "REGION_RATING_CLIENT_W_CITY",
    
    # External credit scores
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    
    # Bureau inquiries
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

In [8]:
missing_initial_features = [
    col for col in initial_features
    if col not in X_train.columns
]

if missing_initial_features:
    raise ValueError(
        f"Initial features missing from dataset: {missing_initial_features}"
    )

print("Number of initial features:", len(initial_features))

Number of initial features: 32


In [9]:
X_train_fe = X_train[initial_features].copy()
X_test_fe = X_test[initial_features].copy()

print("Training feature dataset:", X_train_fe.shape)
print("Test feature dataset:", X_test_fe.shape)

Training feature dataset: (246008, 32)
Test feature dataset: (61503, 32)


In [10]:
feature_summary = pd.DataFrame({
    "dtype": X_train_fe.dtypes.astype(str),
    "missing_count": X_train_fe.isna().sum(),
    "missing_rate": X_train_fe.isna().mean(),
    "n_unique": X_train_fe.nunique(dropna=False)
}).sort_values(
    by="missing_rate",
    ascending=False
)

feature_summary

,dtype,missing_count,missing_rate,n_unique
EXT_SOURCE_1,float64,138595,0.563376,94565
OCCUPATION_TYPE,str,76940,0.312754,19
EXT_SOURCE_3,float64,48805,0.198388,807
AMT_REQ_CREDIT_BUREAU_YEAR,float64,33244,0.135134,25
AMT_REQ_CREDIT_BUREAU_QRT,float64,33244,0.135134,11
AMT_REQ_CREDIT_BUREAU_MON,float64,33244,0.135134,24
AMT_REQ_CREDIT_BUREAU_WEEK,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_DAY,float64,33244,0.135134,10
AMT_REQ_CREDIT_BUREAU_HOUR,float64,33244,0.135134,6
EXT_SOURCE_2,float64,531,0.002158,108826


In [11]:
categorical_features_raw = (
    X_train_fe
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

numerical_features_raw = (
    X_train_fe
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

print("Categorical features:", len(categorical_features_raw))
print(categorical_features_raw)

print("\nNumerical features:", len(numerical_features_raw))
print(numerical_features_raw)

Categorical features: 10
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

Numerical features: 22
['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR']


/var/folders/qk/7q9gfxws52n7zsd9_0fr1_wh0000gn/T/ipykernel_90321/151657628.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


### Initial Feature Review

The selected features include a mixture of numerical and categorical variables.

Several important risk variables contain missing values, especially the external credit scores and occupation information. Missing values will not be dropped automatically because missingness may itself carry information about applicant risk.

Time variables are currently stored as negative day counts and must be converted into interpretable units before modeling.

## 4. Data Cleaning

Several Home Credit variables require business-aware cleaning before feature engineering.

The most important known issue is `DAYS_EMPLOYED = 365243`, which is a special placeholder rather than a valid employment duration.

This placeholder will be converted to missing, while an additional indicator will preserve the information that the applicant had a special employment status.

In [12]:
employment_value_counts = (
    X_train_fe["DAYS_EMPLOYED"]
    .value_counts(dropna=False)
    .head(10)
)

employment_value_counts

DAYS_EMPLOYED
 365243    44143
-199         125
-200         124
-212         124
-229         121
-230         120
-224         120
-215         117
-207         117
-222         117
Name: count, dtype: int64

In [13]:
SPECIAL_EMPLOYED_VALUE = 365243

for dataset in [X_train_fe, X_test_fe]:
    dataset["DAYS_EMPLOYED_SPECIAL_FLAG"] = (
        dataset["DAYS_EMPLOYED"] == SPECIAL_EMPLOYED_VALUE
    ).astype("int8")

    dataset["DAYS_EMPLOYED"] = dataset["DAYS_EMPLOYED"].replace(
        SPECIAL_EMPLOYED_VALUE,
        np.nan
    )

In [14]:
print(
    "Remaining special values in training data:",
    (X_train_fe["DAYS_EMPLOYED"] == SPECIAL_EMPLOYED_VALUE).sum()
)

print(
    "Special employment flag rate:",
    X_train_fe["DAYS_EMPLOYED_SPECIAL_FLAG"].mean()
)

Remaining special values in training data: 0
Special employment flag rate: 0.1794372540730383


In [15]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series
) -> pd.Series:
    """
    Divide two pandas Series while safely handling
    zero and missing denominators.
    """
    denominator_clean = denominator.replace(0, np.nan)

    result = numerator / denominator_clean

    return result.replace([np.inf, -np.inf], np.nan)

In [16]:
test_numerator = pd.Series([10, 20, 30])
test_denominator = pd.Series([2, 0, np.nan])

safe_divide(test_numerator, test_denominator)

0    5.0
1    NaN
2    NaN
dtype: float64

## 5. Domain Feature Engineering

The following features are created using credit-risk and affordability logic.

The transformations are implemented in one reusable function so that identical logic is applied to both training and test data.

In [17]:
def create_domain_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Create interpretable credit-risk features.

    Parameters
    ----------
    data:
        Raw feature DataFrame.

    Returns
    -------
    pd.DataFrame
        A copy of the input data with engineered features.
    """
    data = data.copy()

    # --------------------------------------------------
    # Applicant age and employment history
    # --------------------------------------------------

    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25

    data["EMPLOYMENT_YEARS"] = (
        -data["DAYS_EMPLOYED"] / 365.25
    )

    data["REGISTRATION_YEARS"] = (
        -data["DAYS_REGISTRATION"] / 365.25
    )

    data["ID_PUBLISH_YEARS"] = (
        -data["DAYS_ID_PUBLISH"] / 365.25
    )

    # --------------------------------------------------
    # Affordability features
    # --------------------------------------------------

    data["CREDIT_INCOME_RATIO"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_INCOME_TOTAL"]
    )

    data["ANNUITY_INCOME_RATIO"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_INCOME_TOTAL"]
    )

    data["CREDIT_ANNUITY_RATIO"] = safe_divide(
        data["AMT_CREDIT"],
        data["AMT_ANNUITY"]
    )

    data["INCOME_PER_PERSON"] = safe_divide(
        data["AMT_INCOME_TOTAL"],
        data["CNT_FAM_MEMBERS"]
    )

    data["CREDIT_PER_PERSON"] = safe_divide(
        data["AMT_CREDIT"],
        data["CNT_FAM_MEMBERS"]
    )

    # --------------------------------------------------
    # Loan structure features
    # --------------------------------------------------

    data["GOODS_CREDIT_RATIO"] = safe_divide(
        data["AMT_GOODS_PRICE"],
        data["AMT_CREDIT"]
    )

    data["CREDIT_GOODS_DIFFERENCE"] = (
        data["AMT_CREDIT"] - data["AMT_GOODS_PRICE"]
    )

    data["ANNUITY_GOODS_RATIO"] = safe_divide(
        data["AMT_ANNUITY"],
        data["AMT_GOODS_PRICE"]
    )

    # --------------------------------------------------
    # Employment stability features
    # --------------------------------------------------

    data["EMPLOYMENT_AGE_RATIO"] = safe_divide(
        data["EMPLOYMENT_YEARS"],
        data["AGE_YEARS"]
    )

    # --------------------------------------------------
    # External score features
    # --------------------------------------------------

    ext_source_cols = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    data["EXT_SOURCE_MEAN"] = data[
        ext_source_cols
    ].mean(axis=1)

    data["EXT_SOURCE_MIN"] = data[
        ext_source_cols
    ].min(axis=1)

    data["EXT_SOURCE_MAX"] = data[
        ext_source_cols
    ].max(axis=1)

    data["EXT_SOURCE_STD"] = data[
        ext_source_cols
    ].std(axis=1)

    data["EXT_SOURCE_MISSING_COUNT"] = (
        data[ext_source_cols]
        .isna()
        .sum(axis=1)
        .astype("int8")
    )

    # --------------------------------------------------
    # Bureau inquiry features
    # --------------------------------------------------

    bureau_inquiry_cols = [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
        "AMT_REQ_CREDIT_BUREAU_QRT",
        "AMT_REQ_CREDIT_BUREAU_YEAR",
    ]

    data["BUREAU_INQUIRY_TOTAL"] = (
        data[bureau_inquiry_cols]
        .sum(axis=1, min_count=1)
    )

    data["BUREAU_INQUIRY_RECENT"] = (
        data[
            [
                "AMT_REQ_CREDIT_BUREAU_HOUR",
                "AMT_REQ_CREDIT_BUREAU_DAY",
                "AMT_REQ_CREDIT_BUREAU_WEEK",
                "AMT_REQ_CREDIT_BUREAU_MON",
            ]
        ]
        .sum(axis=1, min_count=1)
    )

    return data

In [18]:
X_train_fe = create_domain_features(X_train_fe)
X_test_fe = create_domain_features(X_test_fe)

print("Training shape after feature engineering:", X_train_fe.shape)
print("Test shape after feature engineering:", X_test_fe.shape)

Training shape after feature engineering: (246008, 53)
Test shape after feature engineering: (61503, 53)


In [19]:
engineered_features = [
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "REGISTRATION_YEARS",
    "ID_PUBLISH_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_ANNUITY_RATIO",
    "INCOME_PER_PERSON",
    "CREDIT_PER_PERSON",
    "GOODS_CREDIT_RATIO",
    "CREDIT_GOODS_DIFFERENCE",
    "ANNUITY_GOODS_RATIO",
    "EMPLOYMENT_AGE_RATIO",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "EXT_SOURCE_STD",
    "EXT_SOURCE_MISSING_COUNT",
    "BUREAU_INQUIRY_TOTAL",
    "BUREAU_INQUIRY_RECENT",
]

In [20]:
X_train_fe[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
AGE_YEARS,246008.0,43.886423,11.944884,2.050376e+01,33.952088,43.104723,53.861739,6.907324e+01
EMPLOYMENT_YEARS,201865.0,6.529758,6.405822,-0.000000e+00,2.097194,4.511978,8.695414,4.904038e+01
REGISTRATION_YEARS,246008.0,13.660739,9.649712,-0.000000e+00,5.516769,12.334018,20.479124,6.754825e+01
ID_PUBLISH_YEARS,246008.0,8.197537,4.129431,0.000000e+00,4.709103,8.911704,11.767283,1.970431e+01
CREDIT_INCOME_RATIO,246008.0,3.959396,2.687597,4.807615e-03,2.018667,3.268948,5.168514,4.922720e+01
ANNUITY_INCOME_RATIO,245998.0,0.180911,0.094571,2.238846e-04,0.114583,0.162833,0.229000,1.570600e+00
CREDIT_ANNUITY_RATIO,245998.0,21.624291,7.820534,8.036674e+00,15.647004,20.000000,27.099985,4.530508e+01
INCOME_PER_PERSON,246006.0,93113.103330,106834.975810,3.375000e+03,47250.000000,75000.000000,112500.000000,3.900000e+07
CREDIT_PER_PERSON,246006.0,323994.012357,258839.895636,6.750000e+03,135988.500000,256032.000000,439074.000000,4.031032e+06
GOODS_CREDIT_RATIO,245787.0,0.900686,0.096646,1.666667e-01,0.834725,0.893815,1.000000,6.666667e+00


In [21]:
train_infinite_count = np.isinf(
    X_train_fe.select_dtypes(include="number")
).sum().sum()

test_infinite_count = np.isinf(
    X_test_fe.select_dtypes(include="number")
).sum().sum()

print("Infinite values in training data:", train_infinite_count)
print("Infinite values in test data:", test_infinite_count)

Infinite values in training data: 0
Infinite values in test data: 0


In [22]:
train_columns = set(X_train_fe.columns)
test_columns = set(X_test_fe.columns)

only_in_train = train_columns - test_columns
only_in_test = test_columns - train_columns

print("Columns only in training data:", only_in_train)
print("Columns only in test data:", only_in_test)

Columns only in training data: set()
Columns only in test data: set()


In [23]:
X_train_fe[
    [
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
        "EMPLOYMENT_AGE_RATIO"
    ]
].describe()

,AGE_YEARS,EMPLOYMENT_YEARS,EMPLOYMENT_AGE_RATIO
count,246008.000000,201865.000000,201865.000000
mean,43.886423,6.529758,0.156934
std,11.944884,6.405822,0.133635
min,20.503765,-0.000000,-0.000000
25%,33.952088,2.097194,0.056049
50%,43.104723,4.511978,0.118734
75%,53.861739,8.695414,0.219353
max,69.073238,49.040383,0.728811


In [24]:
invalid_age_count = (
    (X_train_fe["AGE_YEARS"] < 18)
    | (X_train_fe["AGE_YEARS"] > 100)
).sum()

invalid_employment_count = (
    X_train_fe["EMPLOYMENT_YEARS"] < 0
).sum()

employment_longer_than_age = (
    X_train_fe["EMPLOYMENT_YEARS"]
    > X_train_fe["AGE_YEARS"]
).sum()

print("Invalid age count:", invalid_age_count)
print("Negative employment years:", invalid_employment_count)
print(
    "Employment years greater than age:",
    employment_longer_than_age
)

Invalid age count: 0
Negative employment years: 0
Employment years greater than age: 0


In [25]:
for dataset in [X_train_fe, X_test_fe]:
    dataset.loc[
        (dataset["AGE_YEARS"] < 18)
        | (dataset["AGE_YEARS"] > 100),
        "AGE_YEARS"
    ] = np.nan

    dataset.loc[
        dataset["EMPLOYMENT_YEARS"] < 0,
        "EMPLOYMENT_YEARS"
    ] = np.nan

    dataset.loc[
        dataset["EMPLOYMENT_YEARS"] > dataset["AGE_YEARS"],
        "EMPLOYMENT_YEARS"
    ] = np.nan

In [26]:
for dataset in [X_train_fe, X_test_fe]:
    dataset["EMPLOYMENT_AGE_RATIO"] = safe_divide(
        dataset["EMPLOYMENT_YEARS"],
        dataset["AGE_YEARS"]
    )

## 6. Missing-Value Features

Missingness may contain useful information because some applicants have less complete financial, employment, or credit-history records.

Missing indicators are created without replacing the original missing values.

In [27]:
for dataset in [X_train_fe, X_test_fe]:
    dataset["TOTAL_MISSING_COUNT"] = (
        dataset.isna()
        .sum(axis=1)
        .astype("int16")
    )

    dataset["TOTAL_MISSING_RATE"] = (
        dataset.isna()
        .mean(axis=1)
    )

In [28]:
X_train_fe[
    [
        "EXT_SOURCE_MISSING_COUNT",
        "TOTAL_MISSING_COUNT",
        "TOTAL_MISSING_RATE"
    ]
].describe()

,EXT_SOURCE_MISSING_COUNT,TOTAL_MISSING_COUNT,TOTAL_MISSING_RATE
count,246008.000000,246008.000000,246008.000000
mean,0.763922,2.821103,0.052243
std,0.649419,3.723014,0.068945
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,1.000000,1.000000,0.018519
75%,1.000000,5.000000,0.092593
max,3.000000,19.000000,0.351852


In [29]:
feature_target_review = (
    X_train_fe[engineered_features]
    .assign(TARGET=y_train.values)
    .groupby("TARGET")
    .mean(numeric_only=True)
    .T
)

feature_target_review.columns = [
    "TARGET_0_MEAN",
    "TARGET_1_MEAN"
]

feature_target_review["ABSOLUTE_DIFFERENCE"] = (
    feature_target_review["TARGET_1_MEAN"]
    - feature_target_review["TARGET_0_MEAN"]
)

feature_target_review[
    "RELATIVE_DIFFERENCE"
] = safe_divide(
    feature_target_review["ABSOLUTE_DIFFERENCE"],
    feature_target_review["TARGET_0_MEAN"].abs()
)

feature_target_review.sort_values(
    "ABSOLUTE_DIFFERENCE",
    key=lambda x: x.abs(),
    ascending=False
)

,TARGET_0_MEAN,TARGET_1_MEAN,ABSOLUTE_DIFFERENCE,RELATIVE_DIFFERENCE
CREDIT_PER_PERSON,325815.426935,303253.548051,-22561.878883,-0.069247
CREDIT_GOODS_DIFFERENCE,60286.350945,68836.601144,8550.250199,0.141827
INCOME_PER_PERSON,93283.377693,91174.187608,-2109.190086,-0.022611
AGE_YEARS,44.167717,40.683288,-3.484430,-0.078891
EMPLOYMENT_YEARS,6.678030,4.966719,-1.711310,-0.256260
REGISTRATION_YEARS,13.781205,12.288987,-1.492218,-0.108279
CREDIT_ANNUITY_RATIO,21.701282,20.747626,-0.953655,-0.043945
ID_PUBLISH_YEARS,8.258212,7.506630,-0.751582,-0.091010
EXT_SOURCE_MIN,0.409760,0.282303,-0.127458,-0.311055
EXT_SOURCE_MEAN,0.518977,0.396857,-0.122120,-0.235310


## Feature Engineering Summary

The feature-engineering workflow completed the following steps:

- Replaced the special `DAYS_EMPLOYED = 365243` placeholder with missing values
- Preserved the special employment status using an indicator variable
- Converted negative day variables into interpretable year-based variables
- Created affordability and loan-structure ratios
- Created household-adjusted income and credit features
- Combined external credit scores into summary and missingness features
- Aggregated bureau inquiries across multiple time windows
- Added applicant-level missing-value counts and rates
- Applied identical transformations to training and test data
- Validated that no infinite values were created
- Confirmed that training and test datasets contain identical feature columns

No imputation, encoding, scaling, resampling, or model fitting has been performed yet.

In [30]:
print("Final training feature shape:", X_train_fe.shape)
print("Final test feature shape:", X_test_fe.shape)

print("\nTraining missing values:")
print(X_train_fe.isna().sum().sum())

print("\nTest missing values:")
print(X_test_fe.isna().sum().sum())

Final training feature shape: (246008, 55)
Final test feature shape: (61503, 55)

Training missing values:
694014

Test missing values:
174245


## 7. Feature-Type Definition

The modeling features are divided into numerical and categorical groups.

Numerical features will use median imputation and standard scaling.

Categorical features will use most-frequent imputation and one-hot encoding.

In [31]:
categorical_features = (
    X_train_fe
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

numerical_features = (
    X_train_fe
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

print("Number of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

Number of categorical features: 10
Number of numerical features: 45


/var/folders/qk/7q9gfxws52n7zsd9_0fr1_wh0000gn/T/ipykernel_90321/1202154231.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object", "category"])


In [32]:
print("Categorical features:")
for feature in categorical_features:
    print("-", feature)

print("\nNumerical features:")
for feature in numerical_features:
    print("-", feature)

Categorical features:
- NAME_CONTRACT_TYPE
- CODE_GENDER
- FLAG_OWN_CAR
- FLAG_OWN_REALTY
- NAME_INCOME_TYPE
- NAME_EDUCATION_TYPE
- NAME_FAMILY_STATUS
- NAME_HOUSING_TYPE
- OCCUPATION_TYPE
- ORGANIZATION_TYPE

Numerical features:
- CNT_CHILDREN
- CNT_FAM_MEMBERS
- AMT_INCOME_TOTAL
- AMT_CREDIT
- AMT_ANNUITY
- AMT_GOODS_PRICE
- DAYS_BIRTH
- DAYS_EMPLOYED
- DAYS_REGISTRATION
- DAYS_ID_PUBLISH
- REGION_POPULATION_RELATIVE
- REGION_RATING_CLIENT
- REGION_RATING_CLIENT_W_CITY
- EXT_SOURCE_1
- EXT_SOURCE_2
- EXT_SOURCE_3
- AMT_REQ_CREDIT_BUREAU_HOUR
- AMT_REQ_CREDIT_BUREAU_DAY
- AMT_REQ_CREDIT_BUREAU_WEEK
- AMT_REQ_CREDIT_BUREAU_MON
- AMT_REQ_CREDIT_BUREAU_QRT
- AMT_REQ_CREDIT_BUREAU_YEAR
- DAYS_EMPLOYED_SPECIAL_FLAG
- AGE_YEARS
- EMPLOYMENT_YEARS
- REGISTRATION_YEARS
- ID_PUBLISH_YEARS
- CREDIT_INCOME_RATIO
- ANNUITY_INCOME_RATIO
- CREDIT_ANNUITY_RATIO
- INCOME_PER_PERSON
- CREDIT_PER_PERSON
- GOODS_CREDIT_RATIO
- CREDIT_GOODS_DIFFERENCE
- ANNUITY_GOODS_RATIO
- EMPLOYMENT_AGE_RATIO
- EXT_S

In [34]:
classified_features = set(
    categorical_features + numerical_features
)

unclassified_features = (
    set(X_train_fe.columns) - classified_features
)

print("Unclassified features:", unclassified_features)

Unclassified features: set()


In [35]:
categorical_cardinality = (
    X_train_fe[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .to_frame("n_unique")
)

categorical_cardinality

,n_unique
ORGANIZATION_TYPE,58
OCCUPATION_TYPE,19
NAME_INCOME_TYPE,8
NAME_FAMILY_STATUS,6
NAME_HOUSING_TYPE,6
NAME_EDUCATION_TYPE,5
CODE_GENDER,3
NAME_CONTRACT_TYPE,2
FLAG_OWN_CAR,2
FLAG_OWN_REALTY,2


In [36]:
HIGH_CARDINALITY_THRESHOLD = 50

high_cardinality_features = (
    categorical_cardinality[
        categorical_cardinality["n_unique"]
        > HIGH_CARDINALITY_THRESHOLD
    ]
)

high_cardinality_features

,n_unique
ORGANIZATION_TYPE,58


## 8. Preprocessing Pipeline

The preprocessing pipeline is fit only on the training data.

Numerical features are processed using:

1. Median imputation
2. Missing-value indicators
3. Standard scaling

Median imputation is preferred over mean imputation because many financial variables are right-skewed and contain extreme values.

In [37]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [38]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

In [39]:
from sklearn.preprocessing import OneHotEncoder

In [40]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        ),
    ]
)

In [41]:
from sklearn.compose import ColumnTransformer

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

### Fit the Preprocessor

The preprocessor is fit only on the training data.

The learned medians, category frequencies, scaling parameters, and one-hot categories are then applied unchanged to the test data.

In [44]:
X_train_processed = preprocessor.fit_transform(
    X_train_fe
)

X_test_processed = preprocessor.transform(
    X_test_fe
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)
assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

print("Processed train/test feature counts match.")

Processed training shape: (246008, 183)
Processed test shape: (61503, 183)
Processed train/test feature counts match.


In [45]:
processed_feature_names = (
    preprocessor.get_feature_names_out()
)

print(
    "Number of processed features:",
    len(processed_feature_names)
)

processed_feature_names[:30]

Number of processed features: 183


array(['numerical__CNT_CHILDREN', 'numerical__CNT_FAM_MEMBERS',
       'numerical__AMT_INCOME_TOTAL', 'numerical__AMT_CREDIT',
       'numerical__AMT_ANNUITY', 'numerical__AMT_GOODS_PRICE',
       'numerical__DAYS_BIRTH', 'numerical__DAYS_EMPLOYED',
       'numerical__DAYS_REGISTRATION', 'numerical__DAYS_ID_PUBLISH',
       'numerical__REGION_POPULATION_RELATIVE',
       'numerical__REGION_RATING_CLIENT',
       'numerical__REGION_RATING_CLIENT_W_CITY',
       'numerical__EXT_SOURCE_1', 'numerical__EXT_SOURCE_2',
       'numerical__EXT_SOURCE_3', 'numerical__AMT_REQ_CREDIT_BUREAU_HOUR',
       'numerical__AMT_REQ_CREDIT_BUREAU_DAY',
       'numerical__AMT_REQ_CREDIT_BUREAU_WEEK',
       'numerical__AMT_REQ_CREDIT_BUREAU_MON',
       'numerical__AMT_REQ_CREDIT_BUREAU_QRT',
       'numerical__AMT_REQ_CREDIT_BUREAU_YEAR',
       'numerical__DAYS_EMPLOYED_SPECIAL_FLAG', 'numerical__AGE_YEARS',
       'numerical__EMPLOYMENT_YEARS', 'numerical__REGISTRATION_YEARS',
       'numerical__ID_PUBL

In [58]:
clean_feature_names = [
    feature_name
    .replace("numerical__", "")
    .replace("categorical__", "")
    for feature_name in processed_feature_names
]

clean_feature_names[:30]

['CNT_CHILDREN',
 'CNT_FAM_MEMBERS',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'REGION_POPULATION_RELATIVE',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'AMT_REQ_CREDIT_BUREAU_HOUR',
 'AMT_REQ_CREDIT_BUREAU_DAY',
 'AMT_REQ_CREDIT_BUREAU_WEEK',
 'AMT_REQ_CREDIT_BUREAU_MON',
 'AMT_REQ_CREDIT_BUREAU_QRT',
 'AMT_REQ_CREDIT_BUREAU_YEAR',
 'DAYS_EMPLOYED_SPECIAL_FLAG',
 'AGE_YEARS',
 'EMPLOYMENT_YEARS',
 'REGISTRATION_YEARS',
 'ID_PUBLISH_YEARS',
 'CREDIT_INCOME_RATIO',
 'ANNUITY_INCOME_RATIO',
 'CREDIT_ANNUITY_RATIO']

In [59]:
if hasattr(X_train_processed, "data"):
    train_missing_after_processing = np.isnan(
        X_train_processed.data
    ).sum()
else:
    train_missing_after_processing = np.isnan(
        X_train_processed
    ).sum()

if hasattr(X_test_processed, "data"):
    test_missing_after_processing = np.isnan(
        X_test_processed.data
    ).sum()
else:
    test_missing_after_processing = np.isnan(
        X_test_processed
    ).sum()

print(
    "Missing values after training preprocessing:",
    train_missing_after_processing
)

print(
    "Missing values after test preprocessing:",
    test_missing_after_processing
)

Missing values after training preprocessing: 0
Missing values after test preprocessing: 0


In [60]:
assert train_missing_after_processing == 0
assert test_missing_after_processing == 0

In [61]:
if hasattr(X_train_processed, "data"):
    train_infinite_after_processing = np.isinf(
        X_train_processed.data
    ).sum()
else:
    train_infinite_after_processing = np.isinf(
        X_train_processed
    ).sum()

if hasattr(X_test_processed, "data"):
    test_infinite_after_processing = np.isinf(
        X_test_processed.data
    ).sum()
else:
    test_infinite_after_processing = np.isinf(
        X_test_processed
    ).sum()

print(
    "Infinite values after training preprocessing:",
    train_infinite_after_processing
)

print(
    "Infinite values after test preprocessing:",
    test_infinite_after_processing
)

Infinite values after training preprocessing: 0
Infinite values after test preprocessing: 0


In [62]:
print("Training rows:", X_train_processed.shape[0])
print("Training targets:", len(y_train))

print("Test rows:", X_test_processed.shape[0])
print("Test targets:", len(y_test))

Training rows: 246008
Training targets: 246008
Test rows: 61503
Test targets: 61503


In [63]:
assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

print("Feature matrices and targets are aligned.")

Feature matrices and targets are aligned.


In [64]:
import joblib
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [65]:
PREPROCESSOR_PATH = (
    MODEL_DIR / "feature_preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

print(
    "Saved preprocessor to:",
    PREPROCESSOR_PATH
)

Saved preprocessor to: /Users/hxxy/Desktop/找工/credit-risk-decisioning/models/feature_preprocessor.joblib


In [66]:
train_split_output = pd.DataFrame({
    "SK_ID_CURR": train_ids,
    "TARGET": y_train
})

test_split_output = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "TARGET": y_test
})

In [67]:
train_split_output = (
    train_split_output
    .reset_index(drop=True)
)

test_split_output = (
    test_split_output
    .reset_index(drop=True)
)

In [68]:
TRAIN_SPLIT_PATH = (
    PROCESSED_DATA_DIR / "train_split_ids_targets.csv"
)

TEST_SPLIT_PATH = (
    PROCESSED_DATA_DIR / "test_split_ids_targets.csv"
)

train_split_output.to_csv(
    TRAIN_SPLIT_PATH,
    index=False
)

test_split_output.to_csv(
    TEST_SPLIT_PATH,
    index=False
)

print("Saved:", TRAIN_SPLIT_PATH)
print("Saved:", TEST_SPLIT_PATH)

Saved: /Users/hxxy/Desktop/找工/credit-risk-decisioning/data/processed/train_split_ids_targets.csv
Saved: /Users/hxxy/Desktop/找工/credit-risk-decisioning/data/processed/test_split_ids_targets.csv


In [69]:
FEATURE_NAMES_PATH = (
    MODEL_DIR / "processed_feature_names.txt"
)

with open(
    FEATURE_NAMES_PATH,
    "w",
    encoding="utf-8"
) as file:
    for feature_name in clean_feature_names:
        file.write(f"{feature_name}\n")

print(
    "Saved feature names to:",
    FEATURE_NAMES_PATH
)

Saved feature names to: /Users/hxxy/Desktop/找工/credit-risk-decisioning/models/processed_feature_names.txt


In [70]:
preprocessing_summary = pd.Series({
    "raw_training_rows": X_train_fe.shape[0],
    "raw_test_rows": X_test_fe.shape[0],
    "raw_feature_count": X_train_fe.shape[1],
    "categorical_feature_count": len(
        categorical_features
    ),
    "numerical_feature_count": len(
        numerical_features
    ),
    "processed_feature_count": (
        X_train_processed.shape[1]
    ),
    "training_target_rate": y_train.mean(),
    "test_target_rate": y_test.mean(),
    "training_missing_after_processing": (
        train_missing_after_processing
    ),
    "test_missing_after_processing": (
        test_missing_after_processing
    ),
})

preprocessing_summary

raw_training_rows                    246008.000000
raw_test_rows                         61503.000000
raw_feature_count                        55.000000
categorical_feature_count                10.000000
numerical_feature_count                  45.000000
processed_feature_count                 183.000000
training_target_rate                      0.080729
test_target_rate                          0.080728
training_missing_after_processing         0.000000
test_missing_after_processing             0.000000
dtype: float64

## 9. Preprocessing Summary

The feature-engineering and preprocessing workflow is now ready for baseline modeling.

The completed workflow:

- performed a stratified train/test split before learning preprocessing parameters,
- removed the applicant identifier from model features,
- corrected special employment values,
- created interpretable credit-risk and affordability features,
- preserved informative missingness,
- applied median imputation to numerical features,
- applied most-frequent imputation to categorical features,
- standardized numerical features,
- one-hot encoded categorical features,
- handled unseen test categories safely,
- confirmed that no missing or infinite values remained,
- saved the fitted preprocessor and final feature names.

The next notebook will use these transformed features to build and evaluate an interpretable logistic regression baseline.